# Baseline Model Development

This notebook is the first model-development workspace for hourly DK1/DK2 day-ahead price forecasts.

The goal is deliberately modest: make the simple baselines visible, testable, and interpretable before reaching for more expressive models. We use the implemented `dkenergy_forecast` primitives, but we keep the workflow notebook-first: inspect the data, choose a small backtest window, run the models, plot where they work, and plot where they fail.

The baselines in scope are:

- **same hour yesterday**: `LagNaive(lag_hours=24)`
- **same hour yesterday with last-available fallback**: `LagNaive(lag_hours=24, fallback="last_available")`
- **same hour last week**: `LagNaive(lag_hours=168)`
- **rolling median by local hour**: `SeasonalRollingMedian(...)`
- **rolling median by local hour and weekend flag**: a slightly more conditional variant

The notebook never calls Energi Data Service. It reads only the model-ready panel:

```text
data/model_ready/price_panel_hourly_v1.parquet
```

If that file does not exist locally, the notebook creates a small synthetic DK1/DK2 teaching panel so the modeling mechanics and plots remain runnable. Synthetic results are not model evidence; they are a didactic fixture.


<div style="color:#b42318; border-left:4px solid #b42318; padding:0.35rem 0 0.35rem 0.8rem; margin:0.75rem 0;"><strong>Interpretation convention.</strong> Red callouts are modeling interpretation, judgement, or next-step recommendations. Tables, charts, and code cells are the legwork; red text is my read of what the legwork suggests.</div>


In [ ]:
from __future__ import annotations

import json
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
except ImportError:
    sns = None
    plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find project root with pyproject.toml and src/")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dkenergy_forecast.backtesting.horizons import (
    make_daily_origins,
    make_danish_delivery_day_horizon,
)
from dkenergy_forecast.backtesting.rolling_origin import rolling_origin_backtest
from dkenergy_forecast.evaluation.point_metrics import bias, mae, rmse
from dkenergy_forecast.evaluation.value_metrics import cheapest_k_hit_rate
from dkenergy_forecast.io import load_price_panel
from dkenergy_forecast.models.baselines import LagNaive, SeasonalRollingMedian
from dkenergy_forecast.types import add_copenhagen_calendar

PANEL_PATH = PROJECT_ROOT / "data" / "model_ready" / "price_panel_hourly_v1.parquet"
QA_PATH = PROJECT_ROOT / "data" / "model_ready" / "price_panel_hourly_v1.qa.json"

USE_SYNTHETIC_IF_MISSING = True
FORECAST_AT_HOUR_UTC = 10
DEVELOPMENT_BACKTEST_DAYS = 45
K_CHEAPEST_HOURS = 6

print(f"Project root: {PROJECT_ROOT}")
print(f"Panel path: {PANEL_PATH}")
print(f"QA path: {QA_PATH}")


INTERPRETATION_STYLE = (
    "color:#b42318; border-left:4px solid #b42318; "
    "padding:0.35rem 0 0.35rem 0.8rem; margin:0.75rem 0;"
)


def interpretation(text: str) -> None:
    """Display modeler's interpretation in red so it is distinct from notebook legwork."""
    display(Markdown(f'<div style="{INTERPRETATION_STYLE}"><strong>Interpretation.</strong> {text}</div>'))


def format_pct(value: float) -> str:
    return f"{value:.1%}"


## 1. Load the model-ready panel

The forecasting library assumes the data-processing layer has already done the hard contract work: UTC timestamps, Danish local calendar fields, DK1/DK2 shared coverage, no duplicate forecast keys, and `dataset_version` attached to every row.

For exploratory notebooks, it is useful to keep going even before the real panel has been built. The synthetic fallback below is intentionally simple and labeled loudly; it exists only to make the notebook teachable on a fresh checkout.


In [ ]:
def make_synthetic_price_panel(
    start: str = "2024-01-01T00:00:00Z",
    end: str = "2024-05-15T00:00:00Z",
) -> pd.DataFrame:
    """Create a deterministic DK1/DK2 teaching panel with shared hourly UTC coverage."""

    rng = np.random.default_rng(42)
    timestamps = pd.date_range(start=start, end=end, freq="h", inclusive="left", tz="UTC")
    rows = []

    for area, area_offset, noise_scale in [("DK1", 0.0, 14.0), ("DK2", 18.0, 16.0)]:
        local = timestamps.tz_convert("Europe/Copenhagen")
        hour = local.hour.to_numpy()
        day_of_week = local.dayofweek.to_numpy()
        day_index = np.arange(len(timestamps)) / 24

        morning_peak = 55 * np.exp(-0.5 * ((hour - 8) / 2.2) ** 2)
        evening_peak = 75 * np.exp(-0.5 * ((hour - 18) / 3.0) ** 2)
        night_discount = np.where((hour <= 4) | (hour >= 23), -35, 0)
        weekend_discount = np.where(day_of_week >= 5, -28, 0)
        slow_wave = 22 * np.sin(2 * np.pi * day_index / 21)
        area_spread_wave = 9 * np.sin(2 * np.pi * day_index / 9 + (0.8 if area == "DK2" else 0.0))
        noise = rng.normal(0, noise_scale, len(timestamps))

        y = 330 + area_offset + morning_peak + evening_peak + night_discount + weekend_discount + slow_wave + area_spread_wave + noise

        # A few deterministic stress events make the error plots more honest.
        for spike_time, shock in [
            ("2024-02-07T18:00:00Z", 260),
            ("2024-03-14T06:00:00Z", -160),
            ("2024-04-22T19:00:00Z", 210),
        ]:
            spike_time = pd.Timestamp(spike_time)
            if spike_time in timestamps:
                y[timestamps.get_loc(spike_time)] += shock + (20 if area == "DK2" else 0)

        area_frame = pd.DataFrame(
            {
                "unique_id": f"day_ahead_price_{area}",
                "ds_utc": timestamps,
                "area": area,
                "y": y.round(2),
                "dataset_version": "synthetic_demo_v1",
            }
        )
        rows.append(area_frame)

    panel = pd.concat(rows, ignore_index=True)
    panel = add_copenhagen_calendar(panel)
    panel["price_dkk_per_mwh"] = panel["y"]
    panel["price_eur_per_mwh"] = panel["y"] / 7.45
    return panel.sort_values(["area", "ds_utc"]).reset_index(drop=True)


qa = {}
if PANEL_PATH.exists():
    panel = load_price_panel(
        PANEL_PATH,
        QA_PATH if QA_PATH.exists() else None,
        require_final_historical=False,
    )
    if QA_PATH.exists():
        qa = json.loads(QA_PATH.read_text(encoding="utf-8"))
    DATA_SOURCE = "real model-ready EDS panel"
    USING_SYNTHETIC = False
else:
    if not USE_SYNTHETIC_IF_MISSING:
        raise FileNotFoundError(
            f"Missing {PANEL_PATH}. Build the panel or set USE_SYNTHETIC_IF_MISSING=True."
        )
    panel = make_synthetic_price_panel()
    DATA_SOURCE = "synthetic teaching panel"
    USING_SYNTHETIC = True

panel = panel.copy()
panel["ds_utc"] = pd.to_datetime(panel["ds_utc"], utc=True)
panel["ds_local"] = pd.to_datetime(panel["ds_local"], utc=True).dt.tz_convert("Europe/Copenhagen")
panel["local_date_dt"] = pd.to_datetime(panel["local_date"])
panel = panel.sort_values(["area", "ds_utc"]).reset_index(drop=True)

source_note = "Synthetic demo data: use plots to understand mechanics, not market performance." if USING_SYNTHETIC else "Real model-ready data loaded."
display(Markdown(f"Loaded **{len(panel):,}** rows from **{DATA_SOURCE}**. {source_note}"))
display(panel.head(6))


if USING_SYNTHETIC:
    interpretation(
        "This run is using a deterministic synthetic teaching panel. Treat the ranking and error values as a workflow check, "
        "not as evidence about Danish market performance. The notebook will automatically switch to the real model-ready panel once it exists."
    )
else:
    status = qa.get("artifact_status", "missing QA artifact")
    interpretation(
        f"This run uses the real model-ready panel. QA artifact status is `{status}`; keep that status visible before treating results as benchmark evidence."
    )


## 2. Contract checks before modeling

Baseline models are simple, but they are not immune to bad input. The minimum checks below answer three questions:

1. Do we have the columns the forecasting layer expects?
2. Is each `(unique_id, ds_utc)` unique?
3. Do DK1 and DK2 cover the same UTC timestamps?


In [ ]:
required_columns = {
    "unique_id",
    "ds_utc",
    "ds_local",
    "area",
    "y",
    "dataset_version",
    "local_date",
    "local_hour",
    "local_day_of_week",
    "local_month",
    "is_weekend",
    "is_dst",
    "utc_offset_hours",
}

contract_rows = []
contract_rows.append({"check": "required columns present", "passed": required_columns.issubset(panel.columns)})
contract_rows.append({"check": "no duplicate forecast keys", "passed": int(panel.duplicated(["unique_id", "ds_utc"]).sum()) == 0})
contract_rows.append({"check": "no missing target values", "passed": int(panel["y"].isna().sum()) == 0})
contract_rows.append({"check": "two expected price areas", "passed": sorted(panel["area"].unique().tolist()) == ["DK1", "DK2"]})

coverage_sets = {area: set(frame["ds_utc"]) for area, frame in panel.groupby("area")}
shared_coverage = len(coverage_sets) == 2 and len(set(map(frozenset, coverage_sets.values()))) == 1
contract_rows.append({"check": "shared UTC coverage across areas", "passed": shared_coverage})

contract = pd.DataFrame(contract_rows)
display(contract)

coverage = (
    panel.groupby("area")
    .agg(
        rows=("ds_utc", "size"),
        min_utc=("ds_utc", "min"),
        max_utc=("ds_utc", "max"),
        unique_hours=("ds_utc", "nunique"),
        min_price=("y", "min"),
        median_price=("y", "median"),
        max_price=("y", "max"),
    )
    .reset_index()
)
display(coverage)

if not contract["passed"].all():
    display(Markdown("**Stop and fix failed checks before treating these results as model evidence.**"))


if contract["passed"].all():
    interpretation(
        "The panel satisfies the minimum forecasting contract for this notebook. That means any weird baseline behavior below is more likely "
        "to come from the forecast rule or horizon definition than from duplicate keys or area coverage problems."
    )
else:
    interpretation(
        "One or more input-contract checks failed. The notebook can still show mechanics, but model conclusions should be ignored until the panel is fixed."
    )


## 3. First visual read of the target

Before modeling, look at both the time series and the local-hour shape. Baselines mostly exploit repeated seasonal structure, so the obvious question is whether yesterday, last week, or a same-hour median has anything stable to learn.


In [ ]:
plot_days = 21
plot_start = panel["ds_utc"].max() - pd.Timedelta(days=plot_days)
plot_frame = panel.loc[panel["ds_utc"] >= plot_start].copy()

fig, ax = plt.subplots(figsize=(12, 4))
if sns is not None:
    sns.lineplot(data=plot_frame, x="ds_local", y="y", hue="area", ax=ax, linewidth=1.4)
else:
    for area, frame in plot_frame.groupby("area"):
        ax.plot(frame["ds_local"], frame["y"], label=area, linewidth=1.4)
    ax.legend()
ax.set_title(f"Last {plot_days} days in the available panel")
ax.set_xlabel("Danish local delivery time")
ax.set_ylabel("DKK/MWh")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

hourly_profile = (
    panel.groupby(["area", "local_hour"])["y"]
    .agg(mean="mean", median="median")
    .reset_index()
)
fig, ax = plt.subplots(figsize=(10, 4))
if sns is not None:
    sns.lineplot(data=hourly_profile, x="local_hour", y="median", hue="area", marker="o", ax=ax)
else:
    for area, frame in hourly_profile.groupby("area"):
        ax.plot(frame["local_hour"], frame["median"], marker="o", label=area)
    ax.legend()
ax.set_title("Median price by Danish local delivery hour")
ax.set_xlabel("Local hour")
ax.set_ylabel("Median DKK/MWh")
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.show()


hourly_spread = (
    hourly_profile.groupby("area")["median"]
    .agg(lambda values: float(values.max() - values.min()))
    .rename("median_hourly_range")
    .reset_index()
)
display(hourly_spread)
mean_hourly_range = hourly_spread["median_hourly_range"].mean()
interpretation(
    f"The median local-hour curve moves by about {mean_hourly_range:,.1f} DKK/MWh from trough to peak on average across areas. "
    "That is enough structure for same-hour and same-hour-median baselines to be meaningful first opponents for later feature models."
)


## 4. Baselines as transparent forecasting rules

The models below are deliberately not clever. That is the point.

A strong baseline answers: _how much useful structure can we capture without fitting a complex model?_ If a later LightGBM or CatBoost model cannot beat these rules in a leakage-free rolling backtest, it is probably using weak features, an awkward horizon, or an evaluation setup that needs more thought.


In [ ]:
baseline_specs = {
    "same_hour_yesterday_strict": lambda: LagNaive(lag_hours=24),
    "same_hour_yesterday_fallback": lambda: LagNaive(lag_hours=24, fallback="last_available"),
    "same_hour_last_week": lambda: LagNaive(lag_hours=168),
    "rolling_median_local_hour_28d": lambda: SeasonalRollingMedian(
        lookback_days=28,
        seasonal_keys=("local_hour",),
        min_periods=7,
    ),
    "rolling_median_hour_weekend_56d": lambda: SeasonalRollingMedian(
        lookback_days=56,
        seasonal_keys=("local_hour", "is_weekend"),
        min_periods=4,
    ),
}

baseline_table = []
for label, factory in baseline_specs.items():
    model = factory()
    params = {
        key: value
        for key, value in model.__dict__.items()
        if not key.startswith("_") and key not in {"model_name", "model_version"}
    }
    baseline_table.append({"model_label": label, "class": model.__class__.__name__, "parameters": params})

display(pd.DataFrame(baseline_table))


## 5. Choose a compact rolling-origin development window

This is a model-development notebook, not a full benchmark run. We use a recent-ish window large enough to reveal patterns but small enough to rerun quickly while iterating.

Each origin is set at `FORECAST_AT_HOUR_UTC`, and the horizon is the next Danish local delivery day. That means daylight-saving days naturally have 23 or 25 hourly rows per area.


In [ ]:
def choose_development_origins(panel: pd.DataFrame, days: int, at_hour_utc: int) -> pd.DataFrame:
    min_origin = (panel["ds_utc"].min() + pd.Timedelta(days=60)).normalize()
    max_origin = (panel["ds_utc"].max() - pd.Timedelta(days=2)).normalize()
    start = max(min_origin, max_origin - pd.Timedelta(days=days))
    end = max_origin + pd.Timedelta(days=1)

    origins = make_daily_origins(panel, start=start, end=end, at_hour_utc=at_hour_utc)
    valid_origins = []
    panel_min = panel["ds_utc"].min()
    panel_max = panel["ds_utc"].max()
    for origin in origins["forecast_origin_utc"]:
        horizon = make_danish_delivery_day_horizon(panel, origin, days_ahead=1)
        if horizon["ds_utc"].min() >= panel_min and horizon["ds_utc"].max() <= panel_max:
            valid_origins.append(origin)

    if not valid_origins:
        raise ValueError("No valid development origins fit inside the panel range.")
    return pd.DataFrame({"forecast_origin_utc": valid_origins})


origins = choose_development_origins(
    panel,
    days=DEVELOPMENT_BACKTEST_DAYS,
    at_hour_utc=FORECAST_AT_HOUR_UTC,
)

origin_summary = pd.DataFrame(
    {
        "n_origins": [len(origins)],
        "first_origin_utc": [origins["forecast_origin_utc"].min()],
        "last_origin_utc": [origins["forecast_origin_utc"].max()],
        "forecast_at_hour_utc": [FORECAST_AT_HOUR_UTC],
    }
)
display(origin_summary)

example_horizon = make_danish_delivery_day_horizon(panel, origins["forecast_origin_utc"].iloc[0], days_ahead=1)
display(example_horizon.groupby(["area", "local_date"]).size().rename("horizon_rows").reset_index())
display(example_horizon.head())
assert "y" not in example_horizon.columns


## 6. Run the rolling-origin backtests

For each origin, the backtester constructs:

```text
history = panel[ds_utc < forecast_origin_utc]
```

The future frame passed to the model contains timestamps and known calendar metadata, but not future `y`. Actual target values are joined only after predictions are produced.


In [ ]:
def delivery_day_horizon_builder(panel: pd.DataFrame, origin: pd.Timestamp) -> pd.DataFrame:
    return make_danish_delivery_day_horizon(panel, origin, days_ahead=1)


prediction_frames = []
for model_label, factory in baseline_specs.items():
    preds = rolling_origin_backtest(
        model_factory=factory,
        panel=panel,
        origins=origins,
        horizon_builder=delivery_day_horizon_builder,
        min_train_rows=24 * 21 * panel["area"].nunique(),
    )
    preds["model_label"] = model_label
    prediction_frames.append(preds)

predictions = pd.concat(prediction_frames, ignore_index=True)
predictions["error"] = predictions["y_pred"] - predictions["y"]
predictions["abs_error"] = predictions["error"].abs()
predictions["squared_error"] = predictions["error"] ** 2
predictions["prediction_missing"] = predictions["y_pred"].isna()

run_summary = (
    predictions.groupby("model_label")
    .agg(
        rows=("ds_utc", "size"),
        origins=("forecast_origin_utc", "nunique"),
        missing_predictions=("prediction_missing", "sum"),
        first_delivery_utc=("ds_utc", "min"),
        last_delivery_utc=("ds_utc", "max"),
    )
    .reset_index()
)
display(run_summary)
display(predictions.head())


strict_rows = run_summary.loc[run_summary["model_label"] == "same_hour_yesterday_strict"]
if not strict_rows.empty:
    strict_missing = strict_rows["missing_predictions"].iloc[0] / strict_rows["rows"].iloc[0]
    interpretation(
        f"The strict 24-hour lag leaves {format_pct(strict_missing)} of horizon rows missing. That is not a bug: "
        "from a day-ahead origin, some of tomorrow's delivery hours would require today's later actuals, which are not available yet."
    )


### A small availability lesson

<div style="color:#b42318; border-left:4px solid #b42318; padding:0.35rem 0 0.35rem 0.8rem; margin:0.75rem 0;"><strong>Interpretation.</strong> The strict same-hour-yesterday baseline may produce missing predictions for part of a day-ahead delivery horizon. That is expected when the forecast origin occurs before all of today's delivery hours are known. The fallback variant shows one conservative way to force complete predictions, but the missing-rate column should stay visible because availability is part of the model contract.</div>


In [ ]:
availability_by_horizon = (
    predictions.groupby(["model_label", "horizon"])["prediction_missing"]
    .mean()
    .reset_index()
)

availability_pivot = availability_by_horizon.pivot(
    index="horizon",
    columns="model_label",
    values="prediction_missing",
)
display(availability_pivot.head(30))

fig, ax = plt.subplots(figsize=(11, 4))
if sns is not None:
    sns.lineplot(data=availability_by_horizon, x="horizon", y="prediction_missing", hue="model_label", marker="o", ax=ax)
else:
    for model_label, frame in availability_by_horizon.groupby("model_label"):
        ax.plot(frame["horizon"], frame["prediction_missing"], marker="o", label=model_label)
    ax.legend()
ax.set_title("Prediction missing-rate by horizon step")
ax.set_xlabel("Horizon step within origin and area")
ax.set_ylabel("Missing prediction rate")
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.show()

late_missing = availability_pivot.get("same_hour_yesterday_strict")
if late_missing is not None:
    first_missing_horizon = late_missing[late_missing > 0].index.min()
    interpretation(
        f"The strict yesterday baseline first becomes unavailable at horizon step {first_missing_horizon}. "
        "This is a useful sanity check: leakage-safe models can be less complete than spreadsheet-style baselines that quietly peek at future actuals."
    )


## 7. Overall model comparison

Start with simple aggregate metrics, then immediately break them down by area and hour. A baseline can look fine overall while failing badly in a specific delivery hour or price area.


In [ ]:

def metric_rows(predictions: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model_label, area), frame in predictions.groupby(["model_label", "area"]):
        rows.append(
            {
                "model_label": model_label,
                "area": area,
                "rows": len(frame),
                "evaluated_rows": int(frame["y_pred"].notna().sum()),
                "mae": mae(frame),
                "rmse": rmse(frame),
                "bias": bias(frame),
                "missing_rate": frame["y_pred"].isna().mean(),
            }
        )
    overall = []
    for model_label, frame in predictions.groupby("model_label"):
        overall.append(
            {
                "model_label": model_label,
                "area": "ALL",
                "rows": len(frame),
                "evaluated_rows": int(frame["y_pred"].notna().sum()),
                "mae": mae(frame),
                "rmse": rmse(frame),
                "bias": bias(frame),
                "missing_rate": frame["y_pred"].isna().mean(),
            }
        )
    return pd.concat([pd.DataFrame(overall), pd.DataFrame(rows)], ignore_index=True)


metrics = metric_rows(predictions).sort_values(["area", "mae", "model_label"])
display(metrics)

fig, ax = plt.subplots(figsize=(11, 4))
plot_metrics = metrics.loc[metrics["area"] != "ALL"].copy()
if sns is not None:
    sns.barplot(data=plot_metrics, x="model_label", y="mae", hue="area", ax=ax)
else:
    plot_metrics.pivot(index="model_label", columns="area", values="mae").plot(kind="bar", ax=ax)
ax.set_title("Baseline MAE by area")
ax.set_xlabel("")
ax.set_ylabel("MAE, DKK/MWh")
ax.tick_params(axis="x", labelrotation=30)
plt.tight_layout()
plt.show()

complete_overall = metrics.loc[(metrics["area"] == "ALL") & (metrics["missing_rate"] == 0)].sort_values("mae")
all_overall = metrics.loc[metrics["area"] == "ALL"].sort_values("mae")
if not complete_overall.empty:
    best_complete = complete_overall.iloc[0]
    best_any = all_overall.iloc[0]
    interpretation(
        f"The best complete-coverage baseline is `{best_complete['model_label']}` with MAE {best_complete['mae']:,.1f}. "
        f"The lowest raw MAE row is `{best_any['model_label']}`, but always read it together with missing-rate; incomplete baselines can look deceptively good."
    )


## 8. Rolling-median parameter poke

The implemented rolling median is intentionally simple, but it has real modeling choices: how much history to use and whether to condition only on local hour or also on weekend/weekday. This quick sweep uses every third origin to stay interactive. It is exploratory, not the official benchmark table.


In [ ]:
sweep_origins = origins.iloc[::3].reset_index(drop=True)
sweep_specs = {}
for lookback_days in [14, 28, 56, 84]:
    sweep_specs[f"median_hour_{lookback_days}d"] = lambda lookback_days=lookback_days: SeasonalRollingMedian(
        lookback_days=lookback_days,
        seasonal_keys=("local_hour",),
        min_periods=7,
    )
    sweep_specs[f"median_hour_weekend_{lookback_days}d"] = lambda lookback_days=lookback_days: SeasonalRollingMedian(
        lookback_days=lookback_days,
        seasonal_keys=("local_hour", "is_weekend"),
        min_periods=4,
    )

sweep_frames = []
for model_label, factory in sweep_specs.items():
    preds = rolling_origin_backtest(
        model_factory=factory,
        panel=panel,
        origins=sweep_origins,
        horizon_builder=delivery_day_horizon_builder,
        min_train_rows=24 * 21 * panel["area"].nunique(),
    )
    preds["model_label"] = model_label
    sweep_frames.append(preds)

sweep_predictions = pd.concat(sweep_frames, ignore_index=True)
sweep_predictions["error"] = sweep_predictions["y_pred"] - sweep_predictions["y"]
sweep_predictions["abs_error"] = sweep_predictions["error"].abs()
sweep_predictions["prediction_missing"] = sweep_predictions["y_pred"].isna()

sweep_metrics = metric_rows(sweep_predictions)
sweep_overall = sweep_metrics.loc[sweep_metrics["area"] == "ALL"].sort_values("mae")
display(sweep_overall)

fig, ax = plt.subplots(figsize=(12, 4))
if sns is not None:
    sns.barplot(data=sweep_overall, x="model_label", y="mae", ax=ax)
else:
    ax.bar(sweep_overall["model_label"], sweep_overall["mae"])
ax.set_title("Rolling-median design sweep on every third origin")
ax.set_xlabel("")
ax.set_ylabel("MAE, DKK/MWh")
ax.tick_params(axis="x", labelrotation=35)
plt.tight_layout()
plt.show()

best_sweep = sweep_overall.iloc[0]
worst_sweep = sweep_overall.iloc[-1]
relative_gap = (worst_sweep["mae"] - best_sweep["mae"]) / best_sweep["mae"]
interpretation(
    f"Best sweep candidate: `{best_sweep['model_label']}` at MAE {best_sweep['mae']:,.1f}. "
    f"The gap from best to worst is {format_pct(relative_gap)} on this quick slice. "
    "If that gap is small on the real panel, do not over-tune this baseline; keep the simplest variant as the benchmark."
)


## 9. One delivery day under the microscope

Aggregate metrics are useful, but model development often improves fastest when you inspect a single forecast horizon. The chart below overlays actual prices and all baseline predictions for one DK1 delivery day.


In [ ]:
example_origin = predictions["forecast_origin_utc"].drop_duplicates().sort_values().iloc[len(origins) // 2]
example_area = "DK1" if "DK1" in predictions["area"].unique() else predictions["area"].iloc[0]
example = predictions.loc[
    (predictions["forecast_origin_utc"] == example_origin)
    & (predictions["area"] == example_area)
].copy()

fig, ax = plt.subplots(figsize=(12, 4))
actual = example.drop_duplicates("ds_utc").sort_values("ds_utc")
ax.plot(actual["ds_local"], actual["y"], color="black", linewidth=2.2, label="actual")
for model_label, frame in example.groupby("model_label"):
    frame = frame.sort_values("ds_utc")
    ax.plot(frame["ds_local"], frame["y_pred"], marker="o", linewidth=1.2, alpha=0.85, label=model_label)
ax.set_title(f"Example delivery day forecast, {example_area}, origin {example_origin.isoformat()}")
ax.set_xlabel("Danish local delivery time")
ax.set_ylabel("DKK/MWh")
ax.legend(loc="best")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

display(
    example[["model_label", "ds_local", "y", "y_pred", "error"]]
    .sort_values(["ds_local", "model_label"])
    .head(16)
)


## 10. Error structure by hour and area

Electricity prices are highly hour-dependent. A baseline that works overnight may fail around morning or evening ramps. This plot turns errors into a shape we can act on.


In [ ]:
hourly_errors = (
    predictions.dropna(subset=["y_pred"])
    .groupby(["model_label", "area", "local_hour"])
    .agg(mae=("abs_error", "mean"), bias=("error", "mean"))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
for ax, area in zip(axes, sorted(hourly_errors["area"].unique())):
    frame = hourly_errors.loc[hourly_errors["area"] == area]
    if sns is not None:
        sns.lineplot(data=frame, x="local_hour", y="mae", hue="model_label", marker="o", ax=ax)
    else:
        for model_label, model_frame in frame.groupby("model_label"):
            ax.plot(model_frame["local_hour"], model_frame["mae"], marker="o", label=model_label)
        ax.legend()
    ax.set_title(f"{area}: MAE by local delivery hour")
    ax.set_xlabel("Local hour")
    ax.set_ylabel("MAE, DKK/MWh")
    ax.set_xticks(range(0, 24, 3))
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
plot_errors = predictions.dropna(subset=["y_pred"]).copy()
if sns is not None:
    sns.boxplot(data=plot_errors, x="model_label", y="error", hue="area", ax=ax, showfliers=False)
else:
    plot_errors.boxplot(column="error", by=["model_label", "area"], ax=ax, rot=30)
ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set_title("Forecast error distribution, outliers hidden for readability")
ax.set_xlabel("")
ax.set_ylabel("Prediction minus actual, DKK/MWh")
ax.tick_params(axis="x", labelrotation=30)
plt.tight_layout()
plt.show()


## 11. Residual overlap and an ensemble sniff test

Before building a more complex model, check whether the simple baselines fail in the same places. If their residuals are highly correlated, averaging them will usually not buy much. If they make different mistakes, a very plain average can be a useful benchmark for later tabular models.


In [ ]:
error_wide = predictions.dropna(subset=["y_pred"]).pivot_table(
    index=["forecast_origin_utc", "area", "ds_utc"],
    columns="model_label",
    values="error",
)
residual_corr = error_wide.corr()
display(residual_corr)

fig, ax = plt.subplots(figsize=(7, 5))
if sns is not None:
    sns.heatmap(residual_corr, vmin=-1, vmax=1, cmap="coolwarm", annot=True, fmt=".2f", ax=ax)
else:
    image = ax.imshow(residual_corr, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(residual_corr.columns)), residual_corr.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(residual_corr.index)), residual_corr.index)
    fig.colorbar(image, ax=ax)
ax.set_title("Residual correlation between baselines")
plt.tight_layout()
plt.show()

complete_labels = (
    metrics.loc[(metrics["area"] == "ALL") & (metrics["missing_rate"] == 0)]
    .sort_values("mae")
    ["model_label"]
    .head(2)
    .tolist()
)

if len(complete_labels) >= 2:
    key_cols = ["forecast_origin_utc", "area", "ds_utc"]
    pred_wide = predictions.loc[predictions["model_label"].isin(complete_labels)].pivot_table(
        index=key_cols,
        columns="model_label",
        values="y_pred",
    ).reset_index()
    actual_cols = key_cols + ["unique_id", "horizon", "ds_local", "local_date", "local_hour", "y", "dataset_version"]
    actuals = predictions.drop_duplicates(key_cols)[actual_cols]
    ensemble = pred_wide.merge(actuals, on=key_cols, how="left")
    ensemble["y_pred"] = ensemble[complete_labels].mean(axis=1)
    ensemble["model_label"] = "mean_of_two_best_complete"
    ensemble["model_name"] = "notebook_average"
    ensemble["model_version"] = "exploratory"
    ensemble_metrics = metric_rows(ensemble)
    comparison = pd.concat(
        [
            metrics.loc[(metrics["area"] == "ALL") & (metrics["model_label"].isin(complete_labels))],
            ensemble_metrics.loc[ensemble_metrics["area"] == "ALL"],
        ],
        ignore_index=True,
    ).sort_values("mae")
    display(comparison)

    ensemble_mae = comparison.loc[comparison["model_label"] == "mean_of_two_best_complete", "mae"].iloc[0]
    best_component_mae = metrics.loc[
        (metrics["area"] == "ALL") & (metrics["model_label"].isin(complete_labels)),
        "mae",
    ].min()
    if ensemble_mae < best_component_mae:
        interpretation(
            f"A plain mean of `{complete_labels[0]}` and `{complete_labels[1]}` beats its best component on this slice. "
            "That is interesting enough to keep as a later benchmark, but not enough to add a library abstraction yet."
        )
    else:
        interpretation(
            f"The plain mean of `{complete_labels[0]}` and `{complete_labels[1]}` does not beat the best component here. "
            "Archive this as a checked idea rather than promoting it to the baseline set."
        )
else:
    interpretation("There are not enough complete-coverage baselines to run the ensemble sniff test.")


## 12. Cheapest-hour usefulness

The project plan includes a simple value question: did the model identify the cheapest hours? This is not a replacement for MAE or pinball loss, but it is a useful first bridge from statistical accuracy to decision usefulness.

Here we ask, for each forecasted delivery day and area, how many of the predicted cheapest `K_CHEAPEST_HOURS` were actually among the realized cheapest hours.


In [ ]:
value_frames = []
for model_label, frame in predictions.groupby("model_label"):
    value = cheapest_k_hit_rate(frame, k=K_CHEAPEST_HOURS)
    value["model_label"] = model_label
    value_frames.append(value)

value_metrics = pd.concat(value_frames, ignore_index=True)
value_summary = (
    value_metrics.groupby("model_label")
    .agg(
        mean_hit_rate=("hit_rate", "mean"),
        median_hit_rate=("hit_rate", "median"),
        mean_available_count=("available_count", "mean"),
        evaluated_days=("hit_rate", "size"),
    )
    .sort_values("mean_hit_rate", ascending=False)
    .reset_index()
)
display(value_summary)

fig, ax = plt.subplots(figsize=(10, 4))
if sns is not None:
    sns.barplot(data=value_summary, x="model_label", y="mean_hit_rate", ax=ax)
else:
    ax.bar(value_summary["model_label"], value_summary["mean_hit_rate"])
ax.set_title(f"Mean cheapest-{K_CHEAPEST_HOURS} hit rate")
ax.set_xlabel("")
ax.set_ylabel("Hit rate")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", labelrotation=30)
plt.tight_layout()
plt.show()


best_value = value_summary.iloc[0]
best_mae_complete = complete_overall.iloc[0] if not complete_overall.empty else None
if best_mae_complete is not None:
    interpretation(
        f"Cheapest-hour selection is a different objective from MAE. Here `{best_value['model_label']}` has the strongest mean hit rate, "
        f"while `{best_mae_complete['model_label']}` is the best complete baseline by MAE. If those differ on the real panel, keep both views in model reviews."
    )


## 13. Worst errors are often more instructive than average errors

A first baseline notebook should end by looking at where the simple models break. These rows are candidates for feature ideas: ramp indicators, weekly effects, area spreads, weather, holidays, or special market conditions.


In [ ]:
worst_errors = (
    predictions.dropna(subset=["y_pred"])
    .nlargest(15, "abs_error")
    [[
        "model_label",
        "area",
        "forecast_origin_utc",
        "ds_local",
        "local_hour",
        "y",
        "y_pred",
        "error",
        "abs_error",
    ]]
    .sort_values("abs_error", ascending=False)
)
display(worst_errors)

fig, ax = plt.subplots(figsize=(12, 4))
worst_day = worst_errors.iloc[0]
worst_context = predictions.loc[
    (predictions["forecast_origin_utc"] == worst_day["forecast_origin_utc"])
    & (predictions["area"] == worst_day["area"])
].copy()
actual = worst_context.drop_duplicates("ds_utc").sort_values("ds_utc")
ax.plot(actual["ds_local"], actual["y"], color="black", linewidth=2.2, label="actual")
for model_label, frame in worst_context.groupby("model_label"):
    frame = frame.sort_values("ds_utc")
    ax.plot(frame["ds_local"], frame["y_pred"], marker="o", linewidth=1.2, alpha=0.85, label=model_label)
ax.set_title(f"Context around largest absolute baseline error: {worst_day['area']}")
ax.set_xlabel("Danish local delivery time")
ax.set_ylabel("DKK/MWh")
ax.legend(loc="best")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


interpretation(
    f"Largest inspected miss: `{worst_day['model_label']}` in {worst_day['area']} at {worst_day['ds_local']} "
    f"with absolute error {worst_day['abs_error']:,.1f} DKK/MWh. Use rows like this as feature prompts, not as reasons to clip the target."
)



## 14. Modeling notes from this pass

Use this section as the handoff to the next modeling iteration.

- The backtest is rolling-origin and target-safe: models see history only before the origin.
- The horizon is an explicit Danish delivery day rather than a hard-coded 24 rows.
- The strict 24-hour lag is intentionally allowed to be incomplete. Its missing-rate is a modeling fact, not a plotting annoyance.
- Rolling-median tuning is worth a quick poke, but if the real-panel gap is small, keep the simpler variant as the official benchmark.
- The ensemble sniff test is deliberately notebook-only. Promote it only if it beats the best component on the real panel and stays interpretable.
- Baselines are intentionally local and inspectable; no external forecasting framework is involved.

When running on the real panel, the next notebook can introduce lag/rolling feature tables and a simple quantile model while preserving the same origin and horizon definitions used here.
